# Matching the JobTtile with the Required Skills for Task B

In this notebook, a step-by-step tutorial is provided for preparing the submission file for the shared task Task B. To achieve this, the data for Task B, hosted on [Zenodo](https://doi.org/10.5281/zenodo.14002665), will be downloaded; a file with the appropriate [submission format](https://talentclef.github.io/talentclef/docs/talentclef-2025/evaluation/) will be prepared, and it will be evaluated using the [task's evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script). Additionally, the provided format is also compatible with the benchmark where the test set data will be uploaded on Codabench.

-----------------------------
TalentCLEF is an initiative to advance Natural Language Processing (NLP) in Human Capital Management (HCM). It aims to create a public benchmark for model evaluation and promote collaboration to develop fair, multilingual, and flexible systems that improve Human Resources (HR) practices across different industries.

This shared-task's inaugural edition is part of the [Conference and Labs of the Evaluation Forum (CLEF)](https://clef2025.clef-initiative.eu/index.php?page=Pages/labs.html), scheduled to be held in Madrid in 2025. If you are interested in registering, you can find registration form [here](https://clef2025-labs-registration.dei.unipd.it/).

<img src="https://github.com/TalentCLEF/talentclef/blob/main/logo_talentclef.png?raw=true" alt="TalentCLEF logo" width="200"/>
<img src="https://talentclef.github.io/talentclef/docs/talentclef-2025/workshop/logo_clef_madrid.png" alt="TalentCLEF logo" width="150"/>


## Imports

- Basics

In [1]:
import os
import sys

- Medium

In [2]:
import pandas as pd
import numpy as np
import json
import subprocess
import ast

- Advanced

In [3]:
import networkx as nx

from sentence_transformers import SentenceTransformer, util
from codecarbon import EmissionsTracker

## Path preparation (root, data, models, results, and evaluations)

In [4]:
root_path="/Users/zia-mac/Documents/projects/data/ir_data/talent_clef/TaskB"

In [5]:
training_data_path = os.path.join(root_path, "training")
validation_data_path=os.path.join(root_path, "validation")
test_data_path=os.path.join(root_path, "test")

In [6]:
models_path=os.path.join(root_path, "models")
results_path=os.path.join(root_path, "results")
evaluations_path=os.path.join(root_path, "evaluations")

In [7]:
sbert_model="all-MiniLM-L6-v2"
sentence_transformer_model=os.path.join(models_path, sbert_model)

## Training Data (Preparation and Modeling)

- **Reading job2skill file**

In [8]:
# Read job2skill file
job2skill = pd.read_csv(os.path.join(training_data_path, 'job2skill.tsv'),
                        sep="\t",
                        names=["job_id","skill_id","rel_type"])
job2skill.head()

,job_id,skill_id,rel_type
0,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/93a68dcb-3dc6...,essential
1,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/05bc7677-5a64...,essential
2,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/860be36a-d19b...,essential
3,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/fed5b267-73fa...,essential
4,http://data.europa.eu/esco/occupation/00030d09...,http://data.europa.eu/esco/skill/f64fe2c2-d090...,essential


- **Reading Job2Terms**

In [9]:
# Read json files
jobid2terms_path = os.path.join(training_data_path, "jobid2terms.json")
with open(jobid2terms_path, 'r') as file:
    jobid2terms = json.load(file)
#print (jobid2terms['http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34'])

In [10]:
for jobid in jobid2terms:
    print (jobid)
    print (jobid2terms[jobid])
    break

http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34
['technical director', 'technical and operations director', 'head of technical', 'director of technical arts', 'head of technical department', 'technical supervisor', 'technical manager']


- **Reading Skilled2Terms**

In [11]:
# Read json files
skillid2terms_path = os.path.join(training_data_path, "skillid2terms.json")
with open(skillid2terms_path, 'r') as file:
    skillid2terms = json.load(file)
#print (skillid2terms)

In [12]:
for skillid in skillid2terms:
    print (skillid)
    print (skillid2terms[skillid])
    break

http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab
['manage musical staff', 'manage staff of music', 'coordinate duties of musical staff', 'manage music staff', 'direct musical staff']


In [13]:
jobid

'http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34'

In [14]:
skillid

'http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab'

In [15]:
job2skill[job2skill["skill_id"]==skillid]

,job_id,skill_id,rel_type
7208,http://data.europa.eu/esco/occupation/0e216d47...,http://data.europa.eu/esco/skill/0005c151-5b5a...,essential
9926,http://data.europa.eu/esco/occupation/13d8b1be...,http://data.europa.eu/esco/skill/0005c151-5b5a...,essential
96038,http://data.europa.eu/esco/occupation/cf7b89d0...,http://data.europa.eu/esco/skill/0005c151-5b5a...,essential
110573,http://data.europa.eu/esco/occupation/f6803a58...,http://data.europa.eu/esco/skill/0005c151-5b5a...,optional


In [16]:
skillid2terms[skillid]

['manage musical staff',
 'manage staff of music',
 'coordinate duties of musical staff',
 'manage music staff',
 'direct musical staff']

In [17]:
jobid2terms[jobid]

['technical director',
 'technical and operations director',
 'head of technical',
 'director of technical arts',
 'head of technical department',
 'technical supervisor',
 'technical manager']

In [18]:
len(job2skill), len(jobid2terms), len(skillid2terms)

(114699, 3039, 13939)

## Test data (Preparataion and Prediction)

#### Load queries and corpus elements in English from the Validation folder:

In [19]:
test_queries_path = os.path.join(test_data_path, "queries")
test_corpus_elements_path = os.path.join(test_data_path, "corpus_elements")

In [20]:
test_queries = pd.read_csv(test_queries_path,sep="\t")
test_corpus_elements = pd.read_csv(test_corpus_elements_path, sep="\t")

In [21]:
len(test_queries), len(test_corpus_elements)

(1520, 1986)

#### Transform `skill_aliases` column to a list of strings:

In [22]:
test_corpus_elements["skill_aliases"] = test_corpus_elements["skill_aliases"].apply(lambda x: ast.literal_eval(x))

#### Approaches of Skill term expansion
 

##### **Approach A (doc_A): Extracting skill related terms**

In [23]:
def related_skill_terms(terms_list):
    return " ".join(terms_list)

In [24]:
test_corpus_elements["doc_A"] = test_corpus_elements["skill_aliases"].apply(lambda x: related_skill_terms(x))

In [25]:
print(len(test_corpus_elements))
test_corpus_elements.columns

1986


Index(['c_id', 'esco_uri', 'skill_aliases', 'doc_A'], dtype='object')

##### **Approach B (doc_B): Extracting terms of jobid associated with the skill**

In [26]:
def get_essential_jobids(skill_id):
    return list(job2skill[(job2skill["skill_id"]==skill_id) & (job2skill["rel_type"]=="essential")]["job_id"])
    

In [27]:
def related_jobid_terms(skill_id):
    related_essential_jobids = get_essential_jobids(skill_id)
    terms_B = []
    for related_jobid in related_essential_jobids:
        related_jobid_terms = jobid2terms[related_jobid]
        terms_B.extend(related_jobid_terms)
    return " ".join(terms_B)

In [28]:
test_corpus_elements["doc_B"] = test_corpus_elements["esco_uri"].apply(lambda x: related_jobid_terms(x))

In [29]:
print (len(test_corpus_elements))
test_corpus_elements.columns

1986


Index(['c_id', 'esco_uri', 'skill_aliases', 'doc_A', 'doc_B'], dtype='object')

##### **Approach C (doc_C): Extracting terms of related skill of related jobid**
  - Skill_id -> {J_1, J_2, ..., J_N} :: J_1-> {S_1, S_2, ..., S_M} :: S_1 -> Terms

In [30]:
def get_essential_skillids(job_id):
    return list(job2skill[(job2skill["job_id"]==job_id) & (job2skill["rel_type"]=="essential")]["skill_id"])       

In [31]:
def related_job2skill_terms(skill_id):
    related_essential_jobids = get_essential_jobids(skill_id)
    
    related_skillids = []
    for related_jobid in related_essential_jobids:
        skill_ids = get_essential_skillids(related_jobid)
        skill_ids.remove(skill_id)
        related_skillids.extend(skill_ids)
    
    terms_C = []
    for related_skillid in related_skillids:
        related_skillid_terms = skillid2terms[related_skillid]
        terms_C.extend(related_skillid_terms)

    return " ".join(terms_C)

In [32]:
test_corpus_elements["doc_C"] = test_corpus_elements["esco_uri"].apply(lambda x: related_job2skill_terms(x))

In [33]:
print (len(test_corpus_elements))
test_corpus_elements.columns

1986


Index(['c_id', 'esco_uri', 'skill_aliases', 'doc_A', 'doc_B', 'doc_C'], dtype='object')

#### Load simple embedding model:

In [34]:
model = SentenceTransformer("all-MiniLM-L6-v2", token=False)

In [35]:
tracker = EmissionsTracker()
tracker.start_task("all-MiniLM-L6-v2")

##### Generate a mapping dictionary between IDs and texts from query

In [36]:
test_queries_ids = test_queries.q_id.to_list()
test_queries_texts = test_queries.jobtitle.to_list()
test_queries_map = dict(zip(test_queries_ids, test_queries_texts))

#### Creating a mapping dictionary of corpus element texts from each of the approaches to corpus_ids

In [37]:
corpus_ids = test_corpus_elements.c_id.to_list()
corpus_esco_uri = test_corpus_elements.esco_uri.to_list()
print (len(corpus_ids), len(corpus_esco_uri))

1986 1986


##### Approaches A, B, and C

In [38]:
#doc_A
corpus_doc_A_texts = test_corpus_elements.doc_A.to_list()
#map_corpus_doc_A = dict(zip(corpus_ids, corpus_doc_A_texts))
#print (len(corpus_doc_A_texts), len(corpus_ids), len(map_corpus_doc_A))

#doc_B
corpus_doc_B_texts = test_corpus_elements.doc_B.to_list()
#map_corpus_doc_B = dict(zip(corpus_ids, corpus_doc_B_texts))
#print (len(corpus_doc_B_texts), len(corpus_ids), len(map_corpus_doc_B))

#doc_C
corpus_doc_C_texts = test_corpus_elements.doc_C.to_list()
#map_corpus_doc_C = dict(zip(corpus_ids, corpus_doc_C_texts))
#print (len(corpus_doc_C_texts), len(corpus_ids), len(map_corpus_doc_C))

In [39]:
#len(corpus_ids), len(corpus_esco_uri), len(corpus_doc_A_texts), len(map_corpus_doc_A), len(corpus_doc_B_texts), len(map_corpus_doc_B), len(corpus_doc_C_texts), len(map_corpus_doc_C)

- Encode queries and corpus elements:

In [40]:
query_embeddings = model.encode(test_queries_texts, convert_to_tensor=True)
#corpus_embeddings = model.encode(corpus_doc_A_texts, convert_to_tensor=True)

In [41]:
corpus_doc_A_embedding = model.encode(corpus_doc_A_texts, convert_to_tensor=True)
corpus_doc_B_embedding = model.encode(corpus_doc_B_texts, convert_to_tensor=True)
corpus_doc_C_embedding = model.encode(corpus_doc_C_texts, convert_to_tensor=True)

In [42]:
corpus_doc_A_embedding.shape,corpus_doc_B_embedding.shape,corpus_doc_C_embedding.shape 

(torch.Size([1986, 384]), torch.Size([1986, 384]), torch.Size([1986, 384]))

Compute similarities

In [43]:
similarities_query_doc_A = util.cos_sim(query_embeddings, corpus_doc_A_embedding).cpu().numpy()

similarities_query_doc_B = util.cos_sim(query_embeddings, corpus_doc_B_embedding).cpu().numpy()

similarities_query_doc_C = util.cos_sim(query_embeddings, corpus_doc_C_embedding).cpu().numpy()

emissions = tracker.stop_task("all-MiniLM-L6-v2")

In [44]:
similarities_query_doc_A.shape, similarities_query_doc_B.shape, similarities_query_doc_C.shape

((1520, 1986), (1520, 1986), (1520, 1986))

In [45]:
#similarities_query_doc_AB = similarities_query_doc_A + similarities_query_doc_B
similarities_query_doc_AC = similarities_query_doc_A + similarities_query_doc_C
#similarities_query_doc_BC = similarities_query_doc_B + similarities_query_doc_C
#similarities_query_doc_ABC = similarities_query_doc_A + similarities_query_doc_B + similarities_query_doc_C
similarities_query_doc_WABC = similarities_query_doc_A*0.5 + similarities_query_doc_B*0.3 + similarities_query_doc_C*0.2

## Prepare submission file

The submissions must follow the TREC Run File format, including headers in the output file. This means that the fle have 6 space-spearated columns per line, with following information:

- q_id: Query ID.
- Q0: A constant identifier, usually "Q0".
- doc_id: ID of the retrieved document.
- rank: Position of the document in the ranking.
- score: Relevance score assigned by the model.
- tag: Experiment name

In [66]:
import numpy as np

def get_ranked_result_list(similarities_query_doc, corpus_doc, model_name):
    results = []
    results_name = []

    #results.append("q_id Q0 doc_id rank score tag")
    for q_idx, q_id in enumerate(test_queries_ids):
        sorted_indices = np.argsort(-similarities_query_doc[q_idx])
        used_doc_ids = set()
        rank_counter = 0
        for c_idx in sorted_indices:  # Consider the full list.
            doc_id = corpus_ids[c_idx]
            # If doc_id was already processed, go to the next one.
            if doc_id in used_doc_ids:
                continue
            used_doc_ids.add(doc_id)
            rank_counter += 1
    
            query_name = test_queries_map[q_id]
            doc_name = corpus_doc[c_idx]
            score = similarities_query_doc[q_idx, c_idx]
    
            results.append(f"{q_id} Q0 {doc_id} {rank_counter} {score:.4f} {model_name}")
            #results_name.append(f"{query_name} Q0 {doc_name} {rank_counter} {score:.4f} A_model")

    return results, results_name

The list has this structure

In [77]:
results_A, results_A_name = get_ranked_result_list(similarities_query_doc_A, corpus_doc_A_texts, "A_model")
result_approach_A_taskB = os.path.join(results_path, "run_testset_A_taskB")
with open(result_approach_A_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_A))

In [78]:
results_B, results_B_name = get_ranked_result_list(similarities_query_doc_B, corpus_doc_B_texts, "B_model")
result_approach_B_taskB = os.path.join(results_path, "run_testset_B_taskB")
with open(result_approach_B_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_B))


In [79]:
results_C, results_C_name = get_ranked_result_list(similarities_query_doc_C, corpus_doc_C_texts, "C_model")
result_approach_C_taskB = os.path.join(results_path, "run_testset_C_taskB")
with open(result_approach_C_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_C))

In [80]:
#results_AB, results_AB_name = get_ranked_result_list(similarities_query_doc_AB, corpus_doc_A_texts, "AB_model")
#result_approach_AB_taskB = os.path.join(results_path, "test_approach_AB_taskB")
#with open(result_approach_AB_taskB, "w", encoding="utf-8") as f:
#    f.write("\n".join(results_AB))

In [81]:
results_AC, results_AC_name = get_ranked_result_list(similarities_query_doc_AC, corpus_doc_A_texts, "AC_model")
result_approach_AC_taskB = os.path.join(results_path, "run_testset_AC_taskB")
with open(result_approach_AC_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_AC))

In [82]:
#results_BC, results_BC_name = get_ranked_result_list(similarities_query_doc_BC, corpus_doc_B_texts, "BC_model")
#result_approach_BC_taskB = os.path.join(results_path, "test_approach_BC_taskB")
#with open(result_approach_BC_taskB, "w", encoding="utf-8") as f:
#    f.write("\n".join(results_BC))

In [83]:
#results_ABC, results_ABC_name = get_ranked_result_list(similarities_query_doc_ABC, corpus_doc_A_texts, "ABC_model")
#result_approach_ABC_taskB = os.path.join(results_path, "test_approach_ABC_taskB")
#with open(result_approach_ABC_taskB, "w", encoding="utf-8") as f:
#    f.write("\n".join(results_ABC))


In [84]:
results_WABC, results_WABC_name = get_ranked_result_list(similarities_query_doc_WABC, corpus_doc_A_texts, "WABC_model")
result_approach_WABC_taskB = os.path.join(results_path, "run_testset_WABC_taskB")
with open(result_approach_WABC_taskB, "w", encoding="utf-8") as f:
    f.write("\n".join(results_WABC))

In [85]:
emissions_path = os.path.join(results_path, "test_emissions.json")
json.dump(dict(emissions.values), open(emissions_path, "w"), ensure_ascii=False, indent=4)

In [86]:
!zip ~/Documents/projects/data/ir_data/talent_clef/TaskB/results/taskB_testset_WABC.zip ~/Documents/projects/data/ir_data/talent_clef/TaskB/results/run_testset_WABC_taskB

  adding: Users/zia-mac/Documents/projects/data/ir_data/talent_clef/TaskB/results/run_testset_WABC_taskB

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 75%)


### Evaluation

For the evaluation, we will use the official [TalentCLEF evaluation script](https://github.com/TalentCLEF/talentclef25_evaluation_script), which uses the Ranx library under the hood.

First, clone the repo and install the requirements file:

In [ ]:
!git clone https://github.com/TalentCLEF/talentclef25_evaluation_script.git
!pip install -r /content/talentclef25_evaluation_script/requirements.txt


Then, select the Qrels file and the Run file to perform the evaluation.


In [ ]:
qrels_file = os.path.join(root_path, "validation", "qrels.tsv")

run_baseline_file = os.path.join(results_path, "test_baseline_taskB.trec")

run_A_file = os.path.join(results_path, "test_approach_A_taskB.trec")

run_B_file = os.path.join(results_path, "test_approach_B_taskB.trec")

run_C_file = os.path.join(results_path, "test_approach_C_taskB.trec")

run_AB_file = os.path.join(results_path, "test_approach_AB_taskB.trec")

run_AC_file = os.path.join(results_path, "test_approach_AC_taskB.trec")

run_BC_file = os.path.join(results_path, "test_approach_BC_taskB.trec")

run_ABC_file = os.path.join(results_path, "test_approach_ABC_taskB.trec")

run_WABC_file = os.path.join(results_path, "test_approach_WABC_taskB.trec")

In [ ]:
print ("Baseline")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_baseline_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach A")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_A_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach B")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_B_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach C")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_C_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach AB")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_AB_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach AC")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_AC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach BC")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_BC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach ABC")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_ABC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

print ("Approach WABC")
command = ["python", "talentclef25_evaluation_script/talentclef_evaluate.py", "--qrels", qrels_file, "--run", run_WABC_file]
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

In [ ]:
#!python talentclef25_evaluation_script/talentclef_evaluate.py --qrels ~/Documents/projects/data/ir_data/talent_clef/TaskB/validation/qrels.tsv --run ~/Documents/projects/data/ir_data/talent_clef/TaskB/results/evaluation_baseline_taskB.tre

In [ ]:
#model.save(taskB_models_path)